# Lab 55: The Gyroscopic Eye

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 6, *The Embodied Avatar*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-55.ipynb)

**What you will do:** measure your own voluntary reaction time and compare it with the book's figures for the vestibulo-ocular reflex and phone stabilisation, then build a synthetic shaky video and stabilise it with optical flow.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

Shake your head gently in Step 1 and Step 4 -- there is no need to shake it hard or fast to feel
the effect. If rapid head movement makes you feel dizzy or nauseated, stop and rest; this can
happen to people prone to motion sickness or with inner-ear conditions.

## The experiment in the book

In Lab 55 you read text while shaking your head, and separately while keeping your head still
and shaking the book instead. Even though the relative motion between your eyes and the page was
the same in both cases, only the book-shaking version blurred the text. The book explains this
with the vestibulo-ocular reflex (VOR): a fast, three-neuron circuit that rotates your eyes to
exactly cancel head movement, with a latency of only about 8-15 ms, far faster than any voluntary
response (at least 150-200 ms) or a phone's image stabilisation (typically 30-50 ms).

## Record your results

The VOR's 8-15 ms latency is far too fast to time with a stopwatch, but you can measure the
much slower "voluntary response" baseline the book contrasts it with, using the classic
ruler-drop test: have a friend hold a ruler vertically, with its lower end between your open
thumb and finger without touching them, then drop it without warning. Catch it as fast as you
can, and read off the distance it fell before you caught it. Repeat five times.

In [ ]:
import numpy as np

drop_distance_cm = [18, 16, 20, 15, 17]   # example data: replace with your own ruler-drop distances, in cm
g = 981   # cm/s^2
reaction_time_ms = [1000 * np.sqrt(2 * d / g) for d in drop_distance_cm]

results = pd.DataFrame({"trial": range(1, 6), "drop distance (cm)": drop_distance_cm,
                         "reaction time (ms)": np.round(reaction_time_ms, 0)})
display(results)
print(f"Your mean simple reaction time: {np.mean(reaction_time_ms):.0f} ms")

## Compare

Compare your measured reaction time with the book's figures: voluntary responses take at least
150-200 ms, the VOR takes about 8-15 ms, and a phone's image stabilisation takes about 30-50 ms.

In [ ]:
mean_rt = np.mean(reaction_time_ms)
print(f"Your reaction time:                 {mean_rt:.0f} ms")
print(f"VOR latency (book):                  ~8-15 ms")
print(f"Phone stabilisation (book):           ~30-50 ms")
print(f"Voluntary response, minimum (book):   ~150-200 ms")
if mean_rt >= 150:
    print("Your ruler-drop time falls in, or above, the book's typical voluntary-response range -- roughly")
    print("10-20 times slower than the VOR's latency.")
else:
    print("Your ruler-drop time is faster than the book's stated 150-200 ms minimum. Catching a dropped")
    print("ruler is a well-practised, simple reaction that can beat that figure, but it is still vastly")
    print("slower than the VOR's 8-15 ms.")
print(f"Your reaction time is roughly {mean_rt / 11.5:.0f}x the VOR's latency (using 11.5 ms as its midpoint).")

## The AI side

Phone cameras stabilise shaky footage the way your VOR stabilises your vision, but with a
digital shift instead of eye muscles. Below, a synthetic "shaky video" (a pattern jittered
frame by frame) is stabilised using optical flow -- OpenCV's per-pixel estimate of how the
image moved between frames -- which is then used to shift each frame back into alignment.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

np.random.seed(3)
size, n_frames = 120, 20
base = np.zeros((size, size), dtype=np.uint8)
cv2.rectangle(base, (40, 40), (80, 80), 255, -1)
cv2.putText(base, "AB", (30, 65), cv2.FONT_HERSHEY_SIMPLEX, 1, 0, 2)

shifts = np.random.randint(-8, 9, size=(n_frames, 2))   # simulated hand-shake
frames = []
for dx, dy in shifts:
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    frames.append(cv2.warpAffine(base, M, (size, size)))

stabilised = [frames[0]]
for i in range(1, n_frames):
    flow = cv2.calcOpticalFlowFarneback(frames[0], frames[i], None, 0.5, 3, 15, 3, 5, 1.2, 0)
    dx, dy = -flow[..., 0].mean(), -flow[..., 1].mean()   # estimated correction to undo the shift
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    stabilised.append(cv2.warpAffine(frames[i], M, (size, size)))

shaky_var = np.var(np.stack(frames), axis=0).mean()
stable_var = np.var(np.stack(stabilised), axis=0).mean()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.stack(frames).mean(axis=0), cmap="Greys_r")
axes[0].set_title(f"Shaky average (variance {shaky_var:.0f})")
axes[1].imshow(np.stack(stabilised).mean(axis=0), cmap="Greys_r")
axes[1].set_title(f"Stabilised average (variance {stable_var:.0f})")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(f"Pixel variance across frames fell from {shaky_var:.0f} to {stable_var:.0f} after stabilisation")
print(f"({100 * (1 - stable_var / shaky_var):.0f}% reduction) -- a blurrier averaged image means more jitter.")

The stabilised average is visibly sharper than the shaky one, showing the optical-flow
correction doing its job frame by frame -- the same task your VOR performs continuously, and
about a thousand times faster. The analogy has limits: this correction runs after all the
frames are already captured, taking milliseconds of computer time per frame comparison, while
the VOR corrects your eye position within the same fraction of a second the head movement is
still happening, using no explicit "flow" computation at all, only a hard-wired
vestibular-to-eye-muscle circuit.

## Questions to think about

1. Your ruler-drop reaction time is a "voluntary response" in the book's terms. Roughly how many VOR-length time windows (8-15 ms each) fit inside your own reaction time? What does that speed gap buy the VOR that a voluntary correction could not?
2. The optical-flow demo computes a single average shift per frame to correct the whole image. Real head rotation does not just shift the image sideways -- it also introduces some rotation and perspective change. Why might a real phone's stabilisation, and your own VOR, need to handle more than simple left-right, up-down shifts?
3. The book says the cerebellum recalibrates VOR gain over days when someone wears image-reversing prism glasses. If you reversed the sign of the estimated correction (dx, dy) in the code instead of applying it directly, what would you expect to happen to the stabilised video, and how is that similar to a mis-calibrated VOR?
4. Why does the book say the VOR bypasses the cortex entirely, and what would you predict happens to reading-while-head-shaking (Step 1) for someone with vestibular nerve damage, based on that circuit?

## Challenge

Record a short video of yourself walking (or ask a friend to film you) using your phone's
camera, once with any image stabilisation setting turned off and once turned on, if your phone
allows you to toggle it. Rate how shaky each clip looks, from 0 (rock steady) to 5 (very shaky),
and note the setting's name.

In [ ]:
# Example data: replace with your own two clips' ratings.
setting = ["stabilisation off", "stabilisation on"]
shakiness_rating = [4, 1]   # example data: 0 (rock steady) to 5 (very shaky)

results = pd.DataFrame({"setting": setting, "shakiness (0-5)": shakiness_rating})
display(results)
results.plot.barh(x="setting", y="shakiness (0-5)", legend=False, figsize=(6, 2))
plt.xlim(0, 5); plt.tight_layout(); plt.show()
print("A real phone's stabilisation, like the optical-flow demo above, corrects frame by frame after capture")
print("or in near-real time -- but even the fastest phone stabilisation (30-50 ms, per the book) is slower")
print("than your own VOR (8-15 ms).")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-55.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")